# 3. MapReduce

In [1]:
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q http://archive.apache.org/dist/spark/spark-3.1.1/spark-3.1.1-bin-hadoop3.2.tgz
!tar xf spark-3.1.1-bin-hadoop3.2.tgz
!pip install -q findspark


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.1.1-bin-hadoop3.2"
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [3]:
import findspark
findspark.init()
from pyspark.sql import SparkSession
from pyspark import SparkContext, SparkConf
spark = SparkSession.builder.master("local[4]").getOrCreate()
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # Property used to format output tables better
spark
sc = SparkContext.getOrCreate(spark.conf)


In [4]:
num_cores = sc.defaultParallelism
print("Number of CPU cores:", num_cores)


Number of CPU cores: 4


In [5]:
from operator import itemgetter
import numpy as np


# 3.1 Multiplication

In [11]:
# Define the two matrices
matrix_a = np.array(
    [[1, 2],
     [3, 4],
     [5, 6]]
)

matrix_b = np.array(
    [[7, 8, 9],
     [10, 11, 12]]
)

# Cache the number of rows of A and columns of B
row_a = matrix_a.copy().shape[0]
col_b = matrix_b.copy().shape[1]

def numpy_to_spark_matrix(sc, matrix, matrix_name):
    num_rows, num_cols = matrix.shape
    spark_matrix_rdd = sc.parallelize([
        (matrix_name, i, j, matrix[i, j])
        for i in range(num_rows)
        for j in range(num_cols)
    ])
    return spark_matrix_rdd

matrix_a = numpy_to_spark_matrix(sc, matrix_a, "A")
matrix_b = numpy_to_spark_matrix(sc, matrix_b, "B")

# Map function
def map_func(element):
    matrix_index, row, col, value = element
    intermediate_result = []

    if matrix_index == "A":
        for i in range(0, col_b):
            key = f"{row},{i}"
            intermediate_result.append((key, (col, value)))
    else:
        for j in range(0, row_a):
            key = f"{j},{col}"
            intermediate_result.append((key, (row, value)))

    return intermediate_result

# Reduce function
def reduce_func(value_list):
    value_list = sorted(value_list, key=itemgetter(0))
    i = 0
    result = 0

    while i < len(value_list) - 1:
        if value_list[i][0] == value_list[i + 1][0]:
            result += value_list[i][1] * value_list[i + 1][1]
            i += 2
        else:
            i += 1

    return result

# Perform MapReduce
result = matrix_a.union(matrix_b).flatMap(map_func).groupByKey().mapValues(list).mapValues(reduce_func)

# Collect the result matrix
result_matrix = result.collect()

# Organize the result into a 2D array
matrix_size = (row_a, col_b)
output_matrix = [[0.0] * matrix_size[1] for _ in range(matrix_size[0])]

for row_info, value in result_matrix:
    row, col = map(int, row_info.split(","))
    output_matrix[row][col] = value

# Print the result matrix
for row in output_matrix:
    print(row)


[27, 30, 33]
[61, 68, 75]
[95, 106, 117]


# 3.2 Addition

In [12]:
# Define the two matrices
matrix_c = np.array(
    [[7, 8],
     [9, 10],
     [11, 12]]
)

matrix_d = np.array(
    [[1, 2],
     [3, 4],
     [5, 6]]
)

# Cache the number of rows and columns
row_cd = matrix_c.copy().shape[0]
col_cd = matrix_c.copy().shape[1]

def numpy_to_spark_matrix(sc, matrix, matrix_name):
    num_rows, num_cols = matrix.shape
    spark_matrix_rdd = sc.parallelize([
        (matrix_name, i, j, matrix[i, j])
        for i in range(num_rows)
        for j in range(num_cols)
    ])
    return spark_matrix_rdd

matrix_c = numpy_to_spark_matrix(sc, matrix_c, "C")
matrix_d = numpy_to_spark_matrix(sc, matrix_d, "D")

# Map function for addition
def map_add_func(element):
    matrix_index, row, col, value = element
    return (f"{row},{col}", value)

# Reduce function for addition
def reduce_add_func(value_list):
    result = sum(value_list)
    return result

# Perform MapReduce for addition
result_add = matrix_c.union(matrix_d).map(map_add_func).groupByKey().mapValues(list).mapValues(reduce_add_func)

# Collect the result matrix for addition
result_matrix_add = result_add.collect()

# Organize the result into a 2D array for addition
matrix_size_cd = (row_cd, col_cd)
output_matrix_add = [[0.0] * matrix_size_cd[1] for _ in range(matrix_size_cd[0])]

for row_col, value in result_matrix_add:
    row, col = map(int, row_col.split(","))
    output_matrix_add[row][col] = value

# Print the result matrix for addition
for row in output_matrix_add:
    print(row)


[8, 10]
[12, 14]
[16, 18]
